In [1]:
# Borrar caché de modelos gemma
!rm -rf /root/.cache/huggingface/hub/models--google--gemma*

In [2]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.2 MB/s eta 0:00:00


In [3]:
import torch
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

1. Definición del modelo que vamos a utilizar

In [4]:
model_checkpoint = "google/gemma-2-2b-it"
n_labels = 2

2. Configuración de Cuatización a 4-bits

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Cargando tokenizador y modelo base Gemma")

Cargando tokenizador y modelo base Gemma


3. Tokenizador

In [ ]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
from transformers import AutoTokenizer

# 1. Configuración
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# 2. Descarga FORZADA del archivo problemático
# Esto descarga el archivo grande primero de forma aislada
print("Descargando tokenizer.json...")
try:
    hf_hub_download(repo_id=model_checkpoint, filename="tokenizer.json", token=hf_token)
    print("¡Descarga de tokenizer.json completada!")
except Exception as e:
    print(f"Error en la descarga: {e}")

# 3. Carga del Tokenizer (ahora como el archivo ya está en caché local,
# AutoTokenizer lo leerá al instante sin intentar descargarlo de nuevo)
print("Cargando tokenizer desde caché...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, token=hf_token)
print("¡Éxito!")

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

KeyboardInterrupt: 

4. Inicialización del Modelo Base Cuantizado

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=n_labels,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

5. Buscador dinámico de capas para LoRA

In [ ]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    if 'score' in lora_module_names:
        lora_module_names.remove('score')
    return list(lora_module_names)

modules = find_all_linear_names(model)
print(f"Módulos objetivo encontrados para LoRA: {modules}")

6. Configuración de LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=modules,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

7. Preparación del dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ================================
# 1. Carga y preparación de datos
# ================================
import pandas as pd
from sklearn.model_selection import train_test_split
# from datasets import Dataset

print("Cargando el dataset maestro de entrenamiento...")
# Cargamos el TRAIN_MASTER que contiene el 80% de los datos (el test fijo ya está guardado aparte)
df_train_master = pd.read_csv("/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv")
test_df = pd.read_csv('/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv')

# Limpiamos el índice extra (si existe) y renombramos la columna
if "Unnamed: 0" in df_train_master.columns:
    df_train_master = df_train_master.drop(columns=["Unnamed: 0"])
df_train_master = df_train_master.rename(columns={"label_task_3_1_merged": "label"})
df_train_master["label"] = df_train_master["label"].astype(int)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

# División dinámica: 90% Train, 10% Validation (extraído solo del bloque maestro)
# Mantenemos random_state para que la validación sea estable entre pruebas del mismo modelo
train_df, val_df = train_test_split(df_train_master, test_size=0.10, stratify=df_train_master["label"], random_state=42)

print("Distribución fichero de entrenamiento (Train):")
print(train_df['label'].value_counts())

print("\nDistribución fichero de validación (Valid):")
print(val_df['label'].value_counts())

print("\nDistribución fichero de test (estático):")
print(test_df['label'].value_counts())

# train_dataset = Dataset.from_pandas(train_df)
# eval_dataset = Dataset.from_pandas(val_df)

In [ ]:
def tokenize_data(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

train_dataset.reset_format()
valid_dataset.reset_format()

columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names
columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train: columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid: columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)


8. Hiperparámetros del entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Gemma_2B_QLoRA',
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    eval_strategy='steps',
    eval_steps=0.2,              # Evaluamos cada 20%
    save_strategy='steps',
    save_steps=0.2,              # OBLIGATORIO: Mismo valor que eval_steps
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    logging_strategy='steps',
    logging_steps=0.2            # Para ver los logs al mismo tiempo
)

9. Entrenamiento

In [ ]:
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score


In [ ]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics, # ¡Asegúrate de tener esta función cargada en memoria!
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento de Gemma-2-2B...")
trainer.train()